# Let's turn our voice into text, in a notebook

Shoutout to [OpenAI's Whisper](https://github.com/openai/whisper) and the
[ExecuTorch speech-to-text playground](https://executorch.ai/playground/webgpu/#/demo/speech-to-text)
for the inspiration! We're loading the original **Whisper base** PyTorch model
and compiling it here through Turbine and IREE.

Record a short clip, stop, and watch the transcript appear as words are decoded.
You can upload audio or try the included sample too. Python, audio features,
compilation and model execution all happen in this tab.

The CPU and WebGPU notebooks differ only in `DEVICE`. Both use the same FP32
weights, audio features, cached decoder and greedy token selection. Run one
notebook at a time when comparing timings.


In [ ]:
import json
import time
import wave
from pathlib import Path

import numpy as np
import torch
from transformers import WhisperFeatureExtractor, WhisperForConditionalGeneration, WhisperTokenizer
from whisper_graphs import AudioEncoder, DecoderStep, empty_self_cache
from speech_pipeline import WhisperSpeech

DEVICE = 'cpu'
MODEL_DIR = Path('/models/whisper-base')
torch.autograd.set_multithreading_enabled(False)
print(f'PyTorch {torch.__version__} | {DEVICE}')


## 1. Load the original model and a sample clip

We're using `openai/whisper-base`: 72.6 million parameters. Its model and tokenizer
come from Transformers. The original checkpoint and tokenizer files are preloaded
at `/models/whisper-base` in the kernel filesystem. `sources.json` records their
revision and hashes.

The sample is an 11-second excerpt of John F. Kennedy's inaugural address,
distributed with Whisper's tests. The browser panel will also accept your own
recordings and audio files, up to 15 seconds per clip.


In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False,
    attn_implementation='eager').eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
extractor = WhisperFeatureExtractor.from_pretrained(MODEL_DIR, local_files_only=True)

with wave.open('jfk.wav', 'rb') as source:
    assert source.getframerate() == 16000 and source.getnchannels() == 1
    sample_audio = np.frombuffer(source.readframes(source.getnframes()), dtype='<i2').astype(np.float32) / 32768
features = extractor(sample_audio, sampling_rate=16000,
                     return_tensors='pt', return_attention_mask=True)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
print(f'{len(sample_audio) / 16000:.1f} seconds of audio → {tuple(features.input_features.shape)} features')


## 2. Keep an eager PyTorch reference

Before moving the model, run the same audio through standard Transformers generation.
We'll compare the encoder, first decoder logits and complete generated token sequence
with the compiled path. This reference check is separate from our compiled timings.


In [ ]:
reference_started = time.perf_counter()
prefix = [model.generation_config.decoder_start_token_id] + [token for _, token in
    tokenizer.get_decoder_prompt_ids(language='en', task='transcribe', no_timestamps=True)]
with torch.inference_mode():
    expected_generated = model.generate(
        features.input_features, attention_mask=features.attention_mask,
        language='en', task='transcribe', return_timestamps=False,
        do_sample=False, max_new_tokens=124)
    expected_hidden, expected_cross = AudioEncoder(model)(features.input_features)
    expected_logits, _ = DecoderStep(model, 128)(
        torch.tensor([prefix]), torch.arange(len(prefix)), expected_hidden,
        empty_self_cache(model.config, 128, 'cpu'), expected_cross)
expected_text = tokenizer.batch_decode(expected_generated, skip_special_tokens=True)[0].strip()
expected_ids = expected_generated[0].tolist()
if expected_ids[-1] == model.generation_config.eos_token_id:
    expected_ids = expected_ids[:-1]
if expected_ids[:len(prefix)] == prefix:
    expected_ids = expected_ids[len(prefix):]
del expected_cross
print(expected_text)
print(f'Eager reference: {time.perf_counter() - reference_started:.2f} s')


## 3. Compile the encoder and cached decoder

The encoder runs once per clip. Its outputs and cross-attention keys/values stay
on the selected device. The decoder keeps its self-attention cache there too,
so each step processes one new token rather than the whole prefix again.

We read decoder logits back and select the next token in Python, using the
checkpoint's suppression rules. Token selection and text decoding run on the host;
the neural encoder and decoder run on `DEVICE`.

`whisper_graphs.py` wraps the original Transformers modules with tensor-only cache
inputs/outputs. `speech_pipeline.py` contains the async decoding loop. Neither
replaces model layers or changes the checkpoint.


In [ ]:
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 32), name='webgpu')
else:
    gpu = None
model = model.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
speech = WhisperSpeech(model, tokenizer, extractor, device=DEVICE, gpu=gpu)

print('Compiling and checking the sample...', flush=True)
started = time.perf_counter()
cold = await speech.transcribe(sample_audio, retain_reference=True)
torch.testing.assert_close(cold.pop('encoder_reference'), expected_hidden, rtol=3e-3, atol=1e-3)
torch.testing.assert_close(cold.pop('first_logits'), expected_logits, rtol=3e-3, atol=3e-3)
assert cold['token_ids'] == expected_ids, (cold['token_ids'], expected_ids)
assert cold['text'] == expected_text
cold_seconds = time.perf_counter() - started
print(cold['text'])
print(f'Cold compilation + transcription: {cold_seconds:.2f} s')


## 4. Time a completed transcription

The model is now compiled. Run the same clip three times with fresh caches and
report the median completed transcription, with its feature, encoder and decoder
timings. The real-time factor
is processing time divided by clip duration; lower is faster. Recording and initial
package/model downloads are excluded.


In [ ]:
measurements = []
for _ in range(3):
    completed = await speech.transcribe(sample_audio)
    assert completed['text'] == expected_text and completed['token_ids'] == expected_ids
    assert not completed['cancelled'] and not completed['truncated']
    measurements.append(completed)
warm = sorted(measurements, key=lambda result: result['transcription_seconds'])[1]
report = dict(passed=True, model='openai/whisper-base', device=DEVICE, precision='FP32',
              cold_seconds=cold_seconds, audio_seconds=warm['audio_seconds'],
              transcription_seconds=warm['transcription_seconds'],
              real_time_factor=warm['transcription_seconds'] / warm['audio_seconds'],
              transcription_samples_seconds=[result['transcription_seconds'] for result in measurements],
              timings=warm['timings'], transcript=warm['text'])
print(json.dumps(report, indent=2))


## 5. Speak, stop, and read the result

Click **Record microphone**, allow access, and speak. **Stop & transcribe** releases
the microphone and starts recognition. Partial text appears during decoding.
This is record-and-transcribe; it doesn't transcribe while you're still recording.

You can upload a short audio file, listen back, choose its spoken language and
transcribe it. **Cancel** stops after the current model operation. The microphone
and audio decoding use browser audio APIs; the panel doesn't use a browser speech
recognition service or send audio to a server.


In [ ]:
from speech_panel import SpeechPanel

if 'recorder' in globals():
    await recorder.close()
recorder = SpeechPanel(speech.transcribe, sample='jfk.wav', device=DEVICE)


To close the panel from Python, run `await recorder.close()` in a new cell.
Re-running its cell closes the previous instance first.

Whisper can make mistakes, especially with background noise, accents or quiet audio.
Listen back and check important text. The panel skips near-silent clips and reports
when its token limit is reached.
